[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/magrilu/cv-dojo/blob/main/notebooks/camera/camera.ipynb)

# The perspective camera

In this notebook we will build a geometric model of a photograph as a projection
from three-dimensional projective space to the image plane,

$$
\mathbb P^3 \dashrightarrow \mathbb P^2 .
$$

A real camera forms an image through a much richer physical process. Light passes
through a system of lenses, reaches a sensor, and is eventually converted into an
array of pixel values. We will leave all of this aside for now — aperture, focus,
exposure, sensor response, demosaicing, and even lens distortion — and
concentrate on the geometry of image formation alone.

That geometry is much older than photography. Its roots reach back to ancient
optics, and during the Renaissance the same ideas became the basis of linear
perspective: a systematic way of representing a three-dimensional scene on a
two-dimensional surface. Leon Battista Alberti described an image as an
*intersegazione della piramide visiva* — a cross-section of the visual pyramid.

We will take that construction as our starting point: choose a centre of
projection, join it to the points of the scene, and intersect the resulting
visual rays with an image plane.

From there we will gradually build the pinhole camera model. Homogeneous
coordinates will let us encode the entire projection in a single $3\times4$
matrix,

$$
\mathbf x \sim \mathsf P\,\mathbf X ,
$$

defined up to an arbitrary non-zero scale and therefore carrying **eleven degrees
of freedom**.

Before looking at the matrix, however, let us look at the geometry.

In [ ]:
#| echo: false
import sys, subprocess, json
from pathlib import Path

if "google.colab" in sys.modules and not Path("cv-dojo").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/magrilu/cv-dojo.git"], check=True)
    import os
    os.chdir("cv-dojo")

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "cvdojo").exists():
        sys.path.insert(0, str(parent / "src"))
        ROOT = parent
        break

import numpy as np
import matplotlib.pyplot as plt

from cvdojo.house import load_model
from cvdojo.plotting import ACCENT
from cvdojo.scene import set_axes_equal

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": False})

BLUE, GREY, RED = "#3288BD", "0.45", "#C0392B"
DARK = "0.20"   # the world reference frame, which belongs to neither

# One convention, fixed here and used in every figure of the notebook:
#   BLUE   the object,
#   GREY   the rays, and anything shown for reference only,
#   ACCENT the apparatus of projection - the centre, the image plane, the
#          camera frame,
#   DARK   the world reference frame.
# Change the three constants below to restyle every centre of projection at once.
EYE_COLOR, EYE_MARKER, EYE_SIZE = ACCENT, "o", 95

model = load_model()
V3 = {k: np.array(v, float) for k, v in model["vertices"].items()}
IDS = list(V3)
EDGES = model["edges"]
X = np.array([V3[k] for k in IDS])
X_h = np.c_[X, np.ones(len(X))]
index = {k: i for i, k in enumerate(IDS)}

In [ ]:
#| echo: false
# ---------------------------------------------------------------------------
# Camera bookkeeping. Nothing here is conceptual; it is the plumbing that the
# rest of the notebook uses.
# ---------------------------------------------------------------------------

def look_at(centre, target, up=np.array([0.0, 0.0, 1.0])):
    """World-to-camera rotation for a camera at `centre` looking at `target`.

    Camera x points right, y points down in the image, z points forward - the
    convention that makes the projection a plain division by the third
    coordinate.
    """
    Cc, T = np.asarray(centre, float), np.asarray(target, float)
    z = T - Cc; z /= np.linalg.norm(z)
    x = np.cross(z, up); x /= np.linalg.norm(x)
    return np.vstack([x, np.cross(z, x), z])


def make_pose(centre, target):
    R = look_at(centre, target)
    return R, -R @ np.asarray(centre, float)


def K_matrix(fx, fy=None, cx=0.0, cy=0.0, s=0.0):
    return np.array([[fx, s, cx], [0.0, fy if fy else fx, cy], [0.0, 0.0, 1.0]])


def to_camera(pts, R, t):
    """World points expressed in the camera frame."""
    return (np.asarray(R) @ np.asarray(pts, float).T).T + np.asarray(t, float)


def project_points(pts, K, R, t):
    """Pixels and camera-frame coordinates of a set of world points."""
    Xc_ = to_camera(pts, R, t)
    if np.any(Xc_[:, 2] <= 0):
        raise ValueError("some points are on or behind the principal plane")
    p = Xc_[:, :2] / Xc_[:, 2, None]
    q = (K @ np.c_[p, np.ones(len(p))].T).T
    return q[:, :2] / q[:, 2, None], Xc_


def rotation_about(axis, degrees):
    a = np.asarray(axis, float); a /= np.linalg.norm(a)
    th = np.radians(degrees)
    Kx = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
    return np.eye(3) + np.sin(th) * Kx + (1 - np.cos(th)) * (Kx @ Kx)


def draw_eye(ax, centre, label=None, offset=None):
    """The centre of projection. Alberti's eye, and our camera centre."""
    c = np.atleast_1d(np.asarray(centre, float))
    ax.scatter(*c, marker=EYE_MARKER, s=EYE_SIZE, color=EYE_COLOR,
               edgecolors="white", linewidths=1.2, zorder=8)
    if label:
        ax.text(*(c + (0.0 if offset is None else np.asarray(offset, float))),
                label, color=EYE_COLOR, fontsize=12)


def frame_3d(ax, origin, Rt, labels, color=ACCENT, length=2.0, lw=2.0,
             alpha=1.0, ls="-"):
    """Draw a right-handed coordinate frame; Rt has the axes as its columns."""
    o = np.asarray(origin, float)
    for j, name in enumerate(labels):
        end = o + length * np.asarray(Rt)[:, j]
        ax.plot(*zip(o, end), lw=lw, color=color, alpha=alpha, ls=ls)
        if name:
            ax.text(*end, name, fontsize=10, color=color, alpha=alpha)


def wireframe_3d(ax, pts, color=BLUE, lw=1.8, dots=True, alpha=1.0):
    for a, b in EDGES:
        q = pts[[index[a], index[b]]]
        ax.plot(q[:, 0], q[:, 1], q[:, 2], lw=lw, color=color, alpha=alpha)
    if dots:
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=18, color=color,
                   alpha=alpha)


def wireframe_2d(ax, xy_, title=None, width=None, height=None, color=BLUE,
                 lw=2.0, image=True, alpha=1.0, dots=True):
    for a, b in EDGES:
        q = xy_[[index[a], index[b]]]
        ax.plot(q[:, 0], q[:, 1], lw=lw, color=color, alpha=alpha)
    if dots:
        ax.scatter(xy_[:, 0], xy_[:, 1], s=20, color=color, alpha=alpha)
    if width is not None:
        ax.set_xlim(0, width); ax.set_ylim(height, 0)
    elif image:
        ax.invert_yaxis()
    ax.set_aspect("equal")
    if title:
        ax.set_title(title, fontsize=10)


def image_frame(ax, width, height, color=GREY):
    """The border of the digital image, for figures drawn in pixel units."""
    ax.plot([0, width, width, 0, 0], [0, 0, height, height, 0],
            lw=1.0, ls="--", color=color)

In [ ]:
#| echo: false
# The camera used throughout the synthetic part of the notebook.
W_PIX, H_PIX = 960, 720
K = K_matrix(fx=850, cx=W_PIX/2, cy=H_PIX/2)

target = X.mean(axis=0)
C = np.array([7.0, -14.0, 7.0])
R, t = make_pose(C, target)

xy, Xc = project_points(X, K, R, t)
P = K @ np.hstack([R, t.reshape(3, 1)])

## The visual pyramid

Let us start with the collection of 3D points $\{\mathbf{X}_i\}$ that represents
the Origami House, and add a **centre of projection** $\mathbf{C}$: Alberti's
eye, or the optical centre of our ideal camera.

Join $\mathbf{C}$ to all the points of the scene. The resulting family of
concurrent lines is the **visual pyramid**: every scene point $\mathbf X_i$
determines a visual ray through the centre.

To make an image, we intersect these rays with a plane $\phi$. The image point
$\mathbf x_i$ is simply where the ray through $\mathbf X_i$ meets that plane:

$$
\mathbf x_i = \overline{\mathbf C\mathbf X_i}\cap\phi .
$$

Point by point, the three-dimensional scene is cut by the image plane into a
two-dimensional picture: Alberti's *intersegazione della piramide visiva*.

In [ ]:
#| echo: false
#| column: page
#| label: fig-visual-pyramid
#| fig-cap: >-
#|   Alberti's visual pyramid, with the Origami House standing in for his scene.
#|   **Left:** the centre of projection (orange dot), the rays joining it to the
#|   vertices (grey), and the house (blue). **Right:** the marks left where those
#|   rays cross an interposed plane, which is the image.
plane_z = 0.7 * Xc[:, 2].min()
hit = plane_z * Xc[:, :2] / Xc[:, 2, None]

fig = plt.figure(figsize=(14.5, 6.0), layout="constrained")
gs = fig.add_gridspec(1, 2, width_ratios=[1.25, 1.0])

ax = fig.add_subplot(gs[0], projection="3d")
for q in Xc:
    ax.plot(*zip(np.zeros(3), q), lw=0.7, color=GREY, alpha=.85)
wireframe_3d(ax, Xc, dots=False)
half = 1.45 * np.abs(hit).max()
gx, gy = np.meshgrid([-half, half], [-half, half])
ax.plot_surface(gx, gy, np.full_like(gx, plane_z), alpha=.16, color=ACCENT)
for a, b in EDGES:
    q = hit[[index[a], index[b]]]
    ax.plot(q[:, 0], q[:, 1], np.full(2, plane_z), lw=1.5, color=ACCENT)
draw_eye(ax, np.zeros(3))
ax.set_xlabel("$x_c$"); ax.set_ylabel("$y_c$"); ax.set_zlabel("$z_c$")
ax.set_title("the pyramid, and a plane put in its way", fontsize=10)
ax.view_init(elev=11, azim=-67); set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.25)

ax = fig.add_subplot(gs[1])
wireframe_2d(ax, hit, "the intersection: a picture")
ax.axis("off")
plt.show()

## From the visual pyramid to coordinates

To turn the intersection of the visual pyramid into equations, let us choose a
coordinate system.

Place its origin at the centre of projection $\mathbf C$. Let the $z_c$-axis be
perpendicular to the image plane, and let the $x_c$- and $y_c$-axes span
directions parallel to it. If the image plane lies at distance $f$ from the
centre, its equation is simply

$$
z_c=f .
$$

Consider now a scene point $\mathbf{X}_c=(X_c,Y_c,Z_c)^\top$. The visual ray
joining $\mathbf C$ to $\mathbf{X}_c$ meets the image plane at
$\mathbf{x}=(x,y,f)^\top$, and similar triangles in the two coordinate planes
give the coordinates of that intersection:

$$
x=f\frac{X_c}{Z_c},
\qquad
y=f\frac{Y_c}{Z_c}
$$

These are the **perspective projection equations**.

In [ ]:
#| echo: false
#| column: page
#| label: fig-camera-frame
#| fig-cap: >-
#|   The camera reference frame and the two planar sections behind central
#|   projection. **Left:** the scene point $\mathbf{X}_c$ (blue), its visual ray,
#|   and the image point $\mathbf{x}$ where the ray meets the plane $\phi$ at
#|   $z_c=f$. **Centre and right:** the sections in the $x_cz_c$- and
#|   $y_cz_c$-planes, in which the projection equations are two similar
#|   triangles.

# Local names throughout this cell, so that nothing defined earlier is disturbed.
f_d, Zd, Xd, Yd = 1.4, 4.2, 1.55, 1.05
xd, yd = f_d * Xd / Zd, f_d * Yd / Zd

fig = plt.figure(figsize=(15.0, 4.8), layout="constrained")
gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 1.0, 1.0])

# ------------------------------------------------ the frame, in 3D ----------
ax = fig.add_subplot(gs[0], projection="3d")

hx, hy = 1.15, 0.95
gx, gy = np.meshgrid([-hx, hx], [-hy, hy])
ax.plot_surface(gx, gy, np.full_like(gx, f_d), color=ACCENT, alpha=.14,
                shade=False)
corners = np.array([[-hx, -hy, f_d], [hx, -hy, f_d], [hx, hy, f_d],
                    [-hx, hy, f_d], [-hx, -hy, f_d]])
ax.plot(corners[:, 0], corners[:, 1], corners[:, 2], color=ACCENT, lw=1.2,
        alpha=.8)

ax.plot([0, Xd], [0, Yd], [0, Zd], color=BLUE, lw=1.6)
ax.scatter([Xd], [Yd], [Zd], s=38, color=BLUE, depthshade=False, zorder=5)
ax.scatter([xd], [yd], [f_d], s=34, color=ACCENT, depthshade=False, zorder=5)
draw_eye(ax, np.zeros(3))

frame_3d(ax, np.zeros(3), np.eye(3), [r"$x_c$", r"$y_c$", r"$z_c$"],
         color=GREY, length=1.05, lw=1.1)
ax.plot([0, 0], [0, 0], [0, f_d], color=GREY, lw=.9, ls="--", alpha=.8)
ax.text(-.13, -.10, .52*f_d, r"$f$", fontsize=11, color=GREY)
ax.text(-.10, -.10, -.28, r"$\mathbf{C}$", fontsize=12, color=ACCENT)
ax.text(Xd + .10, Yd + .04, Zd, r"$\mathbf{X}_c$", fontsize=12, color=BLUE)
ax.text(xd + .12, yd + .03, f_d, r"$\mathbf{x}$", fontsize=12, color=ACCENT)
ax.text(-hx, hy, f_d + .06, r"$\phi$", fontsize=14, color=ACCENT)

ax.view_init(elev=20, azim=-61)
ax.set_xlim(-1.15, 1.9); ax.set_ylim(-1.15, 1.55); ax.set_zlim(-.15, 4.7)
ax.set_box_aspect((1.05, 1.0, 1.45))
ax.set_axis_off()
ax.set_title("camera reference frame", fontsize=10)

# ------------------------------------------------ the two sections ----------
def draw_section(ax, coord, image_coord, coord_name, image_name):
    ax.axhline(0, lw=.8, color=GREY, alpha=.7)
    top = 1.20 * max(abs(coord), abs(image_coord))
    ax.plot([f_d, f_d], [-.15, top], lw=1.7, color=ACCENT, alpha=.8)
    ax.plot([0, Zd], [0, coord], lw=1.5, color=BLUE)
    ax.plot([Zd, Zd], [0, coord], lw=.9, ls="--", color=GREY, alpha=.7)
    ax.plot([f_d, f_d], [0, image_coord], lw=2.4, color=ACCENT)
    ax.scatter([Zd, f_d], [coord, image_coord], s=[34, 34],
               color=[BLUE, ACCENT], zorder=5)
    draw_eye(ax, [0, 0])

    ax.text(-.10, -.09, r"$\mathbf{C}$", ha="right", va="top", fontsize=11,
            color=ACCENT)
    ax.text(Zd + .08, coord, r"$\mathbf{X}_c$", ha="left", va="center",
            fontsize=11, color=BLUE)
    ax.text(f_d + .08, image_coord, r"$\mathbf{x}$", ha="left", va="center",
            fontsize=11, color=ACCENT)
    ax.text(f_d, -.09, r"$f$", ha="center", va="top", fontsize=10)
    ax.text(Zd, -.09, r"$Z_c$", ha="center", va="top", fontsize=10)
    ax.text(Zd - .10, .52*coord, rf"${coord_name}$", ha="right", va="center",
            fontsize=10, color=BLUE)
    ax.text(f_d - .09, .52*image_coord, rf"${image_name}$", ha="right",
            va="center", fontsize=10, color=ACCENT)
    ax.text(f_d, top*1.02, r"$\phi$", ha="center", va="bottom", fontsize=12,
            color=ACCENT)

    ax.set_xlim(-.25, Zd + .65); ax.set_ylim(-.25, max(coord, image_coord)*1.25)
    ax.set_xlabel(r"$z_c$")
    ax.set_aspect("equal", adjustable="box")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])


ax = fig.add_subplot(gs[1])
draw_section(ax, Xd, xd, "X_c", "x")
ax.set_ylabel(r"$x_c$")
ax.set_title(r"section in the $x_cz_c$-plane", fontsize=10)

ax = fig.add_subplot(gs[2])
draw_section(ax, Yd, yd, "Y_c", "y")
ax.set_ylabel(r"$y_c$")
ax.set_title(r"section in the $y_cz_c$-plane", fontsize=10)
plt.show()

## Depth changes apparent size

The division by $Z_c$ is not just an algebraic detail: it is responsible for one
of the most familiar effects of perspective.

Take two identical Origami Houses and place them side by side, but push one
farther away along the optical axis. Their metric size is exactly the same; only
their position in space differs. Since the projection divides by depth, the
farther house is divided by a larger number and therefore occupies a smaller
region of the image.

In [ ]:
#| echo: false
#| column: page
#| label: fig-depth-and-size
#| fig-cap: >-
#|   Two identical Origami Houses at different depths, the near one solid and the
#|   far one faded. **Left:** in space, where they are the same size. **Right:**
#|   their images, where the farther copy is smaller. Nothing has been scaled;
#|   the shrinking is entirely the division by $Z_c$.
f_two = 0.7 * Xc[:, 2].min()
house_width = np.ptp(Xc[:, 0])

near = Xc + np.array([-0.70*house_width, 0.0, 0.0])
far = Xc + np.array([1.05*house_width, 0.0, 7.0])

to_plane = lambda pts: f_two * pts[:, :2] / pts[:, 2, None]

fig = plt.figure(figsize=(13.5, 5.4), layout="constrained")

ax = fig.add_subplot(121, projection="3d")
wireframe_3d(ax, near, dots=False)
wireframe_3d(ax, far, dots=False, alpha=.40)
draw_eye(ax, np.zeros(3), r"$\mathbf{C}$", offset=[0, 0, -2.0])
ax.set_xlabel("$x_c$"); ax.set_ylabel("$y_c$"); ax.set_zlabel("$z_c$")
ax.set_title("same size, different depth", fontsize=10)
ax.view_init(elev=18, azim=-62); set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.2)

ax = fig.add_subplot(122)
wireframe_2d(ax, to_plane(near), None, lw=1.9, dots=False)
wireframe_2d(ax, to_plane(far), None, lw=1.9, alpha=.40, dots=False)
ax.set_title("their images on the plane $z_c=f$", fontsize=10)
ax.axis("off")
plt.show()

The same factor $1/Z_c$ that produces this familiar effect also makes the
projection nonlinear in Cartesian coordinates. Can we describe the same geometry
with a linear map?

## One extra coordinate makes projection linear

The geometric construction of perspective is centuries older than its modern
algebraic language. In the nineteenth century August Ferdinand Möbius and Julius
Plücker introduced **homogeneous coordinates** for projective geometry, and they
provide exactly what we need here.

Instead of representing the scene point by three Cartesian coordinates, write it
as $\mathbf X=(X_c,Y_c,Z_c,1)^\top$ and its image as $\mathbf x=(x,y,1)^\top$.
Now consider the $3\times4$ matrix

$$
\mathsf P_f=
\begin{bmatrix}
f&0&0&0\\
0&f&0&0\\
0&0&1&0
\end{bmatrix} .
$$

Matrix multiplication gives

$$
\mathsf P_f\mathbf X
=
\begin{bmatrix} fX_c\\ fY_c\\ Z_c \end{bmatrix}
=
Z_c
\begin{bmatrix} fX_c/Z_c\\ fY_c/Z_c\\ 1 \end{bmatrix}
=
Z_c\,\mathbf x ,
$$

where the middle step is nothing but the perspective equations. Hence the whole
construction can be written as

$$
\lambda\mathbf x=\mathsf P_f\mathbf X,
\qquad
\lambda=Z_c .
$$

The division by depth has not disappeared. Homogeneous coordinates have simply
**postponed it**: the projection itself is linear, and the division reappears
only when we choose the representative whose last coordinate is $1$.

### The one point with no image

The matrix $\mathsf P_f$ is $3\times4$, so it has a non-trivial null space, and
it is worth asking what lives there. A vector in the kernel is mapped to
$\mathbf 0$, which is not a point of $\mathbb P^2$ — so a point of the kernel has
no image at all.

For $\mathsf P_f$ the kernel is spanned by $(0,0,0,1)^\top$: the origin of the
camera frame, which is the centre of projection itself. That is exactly right.
Every other point of space determines a visual ray, but the centre determines no
direction, and a camera cannot photograph its own centre.

This is the algebraic shadow of a geometric fact, and it holds for any camera
matrix: the centre is the right null space of $\mathsf P$.

In [ ]:
f_demo = 1.4
P_f = np.array([[f_demo, 0.0, 0.0, 0.0],
                [0.0, f_demo, 0.0, 0.0],
                [0.0, 0.0, 1.0, 0.0]])

C_h = np.linalg.svd(P_f)[2][-1]          # the right null vector
C_h = C_h / np.abs(C_h).max()

print("null vector of P_f:", C_h.round(6))
print("its image P_f C   :", (P_f @ C_h).round(12),
      "  <- not a point of P^2")
print("\nin Cartesian terms this is the origin of the camera frame,")
print("that is, the centre of projection itself.")

## Putting the camera in the world

So far we have made a very convenient choice: the coordinate system was attached
to the camera. The centre of projection was the origin, the $z_c$-axis was the
optical axis, and the image plane was simply $z_c=f$.

But the scene does not have to be described in camera coordinates. For the
Origami House it is more natural to attach a **world reference frame** to the
house itself: origin at a corner of the base, $x_w$ running along its width,
$y_w$ along its depth, and $z_w$ pointing vertically upward.

A scene point is then described by its world coordinates
$\mathbf X_w=(X_w,Y_w,Z_w)^\top$, and before projecting it we only need to
express the same point in the camera frame. Note the phrasing: the point does not
move. Only its description changes.

In [ ]:
#| echo: false
#| column: page
#| label: fig-two-frames
#| fig-cap: >-
#|   One point, two descriptions. The **world frame** (dark) is attached to the
#|   Origami House; the **camera frame** (orange) sits at the centre of
#|   projection with $z_c$ along the optical axis. The highlighted vertex is a
#|   single point of space: the dashed dark segments give its world coordinates,
#|   the dashed orange ray its position as seen from the camera. Passing from one
#|   description to the other is the rigid transformation $[\mathsf R\mid
#|   \mathbf t]$.
key = IDS[len(IDS)//2]
Pw = X[index[key]]

fig = plt.figure(figsize=(14.5, 6.4), layout="constrained")
ax = fig.add_subplot(111, projection="3d")

wireframe_3d(ax, X, dots=False)
frame_3d(ax, np.zeros(3), np.eye(3),
         [r"$x_w$", r"$y_w$", r"$z_w$"], color=DARK, length=12.0)
ax.text(-1.0, 0.6, -1.8, r"$O_w$", fontsize=11, color=DARK)
frame_3d(ax, C, R.T, [r"$x_c$", r"$y_c$", r"$z_c$"], color=ACCENT, length=5.0)
draw_eye(ax, C, r"$\mathbf{C}$", offset=[0, 0, -2.2])

# the chosen point, described twice
ax.scatter(*Pw, s=70, color=RED, zorder=9)
ax.text(*(Pw + np.array([0.4, 0.4, 0.6])), r"$\mathbf{X}$", fontsize=12,
        color=RED)
for seg in [[[0, 0, 0], [Pw[0], 0, 0]],
            [[Pw[0], 0, 0], [Pw[0], Pw[1], 0]],
            [[Pw[0], Pw[1], 0], Pw]]:
    seg = np.array(seg, float)
    ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], lw=1.2, ls="--", color=DARK)
ax.plot(*zip(C, Pw), lw=1.4, ls="--", color=ACCENT)

ax.set_xlabel("$X_w$ [cm]"); ax.set_ylabel("$Y_w$ [cm]")
ax.set_zlabel("$Z_w$ [cm]")
ax.view_init(elev=20, azim=-38); set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.30)
plt.show()

The change of coordinates from the world frame to the camera frame is a rigid
transformation,

$$
\mathbf{X}_c = \mathsf R\,\mathbf X_w+\mathbf t,
$$

or, in homogeneous coordinates,

$$
\begin{bmatrix} \mathbf{X}_c\\ 1 \end{bmatrix}
=
\begin{bmatrix} \mathsf R & \mathbf t\\ \mathbf 0^\top & 1 \end{bmatrix}
\begin{bmatrix} \mathbf X_w\\ 1 \end{bmatrix} .
$$

Here $\mathsf R$ describes the orientation of the world frame as seen by the
camera, while $\mathbf t$ describes its translation in camera coordinates.

If the camera centre has world coordinates $\mathbf C$, then

$$
\mathbf{X}_c
=
\mathsf R(\mathbf X_w-\mathbf C)
=
\mathsf R\mathbf X_w-\mathsf R\mathbf C ,
$$

so that

$$
\mathbf t=-\mathsf R\mathbf C .
$$

The vector $\mathbf t$ is therefore **not the position of the camera** in the
world frame. That position is $\mathbf C$. Mixing up the two is one of the most
reliable ways to spend an afternoon.

In [ ]:
key = IDS[len(IDS)//2]
Xw_pt = X[index[key]]
Xc_pt = R @ Xw_pt + t

print(f"the vertex {key}")
print(f"  in world coordinates : {Xw_pt}")
print(f"  in camera coordinates: {Xc_pt}")
print(f"  distance from C      : {np.linalg.norm(Xw_pt - C):.4f} cm")
print(f"  norm of X_c          : {np.linalg.norm(Xc_pt):.4f} cm   "
      "<- the same, as it must be for a rigid motion\n")
print(f"camera centre C  : {C}")
print(f"translation t    : {t}")
print(f"-R^T t recovers C: {-R.T @ t}")

Projection is now the composition of two maps,
$\mathbf X_w \to \mathbf X_c \to \mathbf x$, that is

$$
\lambda\mathbf x
=
\begin{bmatrix} f&0&0&0\\ 0&f&0&0\\ 0&0&1&0 \end{bmatrix}
\begin{bmatrix} \mathsf R&\mathbf t\\ \mathbf 0^\top&1 \end{bmatrix}
\mathbf X_w ,
$$

and therefore

$$
\lambda\mathbf x
=
\begin{bmatrix} f&0&0\\ 0&f&0\\ 0&0&1 \end{bmatrix}
[\mathsf R\mid\mathbf t]\,\mathbf X_w .
$$

The pair $(\mathsf R,\mathbf t)$ tells us **where the camera is and how it is
oriented with respect to the scene**. These are the **extrinsic parameters**.

### Seeing the extrinsics

Two experiments make the roles of $\mathsf R$ and $\mathbf t$ visible. In both the
house stays exactly where it is; only the camera is touched.

First **rotate** the camera about its own centre. The optical centre does not
move, so $\mathbf C$ is unchanged — yet $\mathbf t=-\mathsf R\mathbf C$ changes,
which is the clearest possible demonstration that $\mathbf t$ is not a position.

Then **translate** the camera, keeping its orientation. Now $\mathsf R$ is
unchanged and the whole apparatus slides sideways.

In [ ]:
#| echo: false
#| column: screen-inset
#| label: fig-extrinsics
#| fig-cap: >-
#|   The two extrinsic motions, with the original camera drawn in grey and the
#|   modified one in orange, and the house fixed in its world frame. **Top:** the
#|   camera turns about its own centre; the two frames share an origin and differ
#|   only in orientation. **Bottom:** the camera slides, keeping its orientation;
#|   the two frames are parallel and their origins differ. The images on the right
#|   are what each camera records, drawn inside the same image rectangle (dashed),
#|   so a part of the house leaving that rectangle has simply left the field of
#|   view.
# Rotating the camera about its OWN axis means a rotation expressed in camera
# coordinates, applied on the left of R. Using the world direction of the camera
# y axis here would produce a roll instead of a pan.
R_rot = rotation_about([0.0, 1.0, 0.0], 15.0) @ R
t_rot = -R_rot @ C
xy_rot, _ = project_points(X, K, R_rot, t_rot)

C_tr = C + np.array([-3.0, 0.0, 1.0])
t_tr = -R @ C_tr
xy_tr, _ = project_points(X, K, R, t_tr)

fig = plt.figure(figsize=(15.5, 9.0), layout="constrained")
gs = fig.add_gridspec(2, 3, width_ratios=[1.5, 1.0, 1.0])

rows = [(0, R_rot, C, "the camera turns about its centre"),
        (1, R, C_tr, "the camera slides, orientation fixed")]

for row, R_new, C_new, ttl in rows:
    ax = fig.add_subplot(gs[row, 0], projection="3d")
    wireframe_3d(ax, X, dots=False)
    frame_3d(ax, C, R.T, ["", "", ""], color=GREY, length=4.6, ls="--")
    frame_3d(ax, C_new, R_new.T, [r"$x_c$", r"$y_c$", r"$z_c$"], color=ACCENT,
             length=4.0)
    ax.scatter(*C, marker=EYE_MARKER, s=EYE_SIZE, color=GREY, zorder=7)
    draw_eye(ax, C_new)
    ax.set_title(ttl, fontsize=10)
    ax.view_init(elev=18, azim=-52); set_axes_equal(ax)
    ax.set_box_aspect((1, 1, 1), zoom=1.45)
    ax.set_axis_off()

for row, pic in [(0, xy_rot), (1, xy_tr)]:
    ax = fig.add_subplot(gs[row, 1])
    wireframe_2d(ax, xy, "before", W_PIX, H_PIX, color=GREY, dots=False)
    image_frame(ax, W_PIX, H_PIX); ax.axis("off")

    ax = fig.add_subplot(gs[row, 2])
    wireframe_2d(ax, pic, "after", W_PIX, H_PIX, dots=False)
    image_frame(ax, W_PIX, H_PIX); ax.axis("off")
plt.show()

In [ ]:
print(f"{'':22s}{'centre C':>34s}{'translation t':>34s}")
for name, R_, C_ in [("original", R, C), ("rotated about C", R_rot, C),
                     ("translated", R, C_tr)]:
    print(f"{name:22s}{str(C_.round(3)):>34s}{str((-R_ @ C_).round(3)):>34s}")
print("\nthe rotated camera has exactly the same centre as the original,")
print("and a different t: the translation vector follows the orientation.")

## From the image plane to pixels

So far we have treated the image as a continuous geometric plane. That is the
setting of Alberti's construction: each visual ray meets the image plane at a
point with continuous coordinates.

A digital image is different. It is a rectangular grid of samples indexed by a
column $u$ and a row $v$, with the origin conventionally at the upper-left
corner, as for the entries of an array. We therefore need one more change of
coordinates, from geometric coordinates on the image plane to **pixel
coordinates**.

Suppose distances in the camera frame are measured in centimetres, so that $x$
and $y$ are centimetres on the image plane. Pixels are not centimetres: if one
pixel has physical width $p_x$ and height $p_y$, a distance on the sensor is
converted into pixels by dividing by those quantities. This is why the focal
length appears in pixel coordinates as

$$
f_x=\frac{f}{p_x},
\qquad
f_y=\frac{f}{p_y} ,
$$

so that $f_x$ and $f_y$ are the focal length expressed in **pixels** rather than
in physical units.

It is convenient to reorganise the construction slightly. Instead of first
projecting onto the physical plane $z_c=f$, intersect the visual ray with the
plane $z_c=1$. This gives the **normalised image coordinates**

$$
\mathbf x_n=(X_c/Z_c,\;Y_c/Z_c,\;1)^\top .
$$

The plane $z_c=1$ is only a convenient reference. We have not moved the sensor,
nor set the physical focal length to one. We have simply factored the projection
into two steps — 3D point, then normalised image, then pixel — with the second
step carrying the focal length, the conversion to pixels, and the choice of image
coordinate system.

### The intrinsics

In homogeneous coordinates that second step is written $\mathbf x\sim\mathsf
K\,\mathbf x_n$, with

$$
\mathsf K=
\begin{bmatrix}
f_x&s&c_x\\
0&f_y&c_y\\
0&0&1
\end{bmatrix} ,
$$

where

- $f_x$ and $f_y$ express the focal length in pixel units;
- $(c_x,c_y)$ is the **principal point**, the pixel at which the optical axis
  meets the image plane;
- $s$ accounts for a possible skew between the two pixel axes, and is zero for
  any sensor that has not been resampled.

The matrix $\mathsf K$ describes how the geometry of the ideal camera is encoded
in the coordinate system of the digital image. Its entries are the **intrinsic
parameters**.

Two of them can be understood by going back to the pyramid and moving the plane.

In [ ]:
#| echo: false
#| column: screen-inset
#| label: fig-focal-length
#| fig-cap: >-
#|   Slicing the same visual pyramid at three distances. **Left:** the centre and
#|   the rays are untouched; only the image plane moves along the optical axis.
#|   **Right:** what each plane records, drawn inside the same image rectangle.
#|   A longer focal length magnifies the picture and narrows the field of view,
#|   but it does not change the pyramid: no new information about the scene has
#|   been gathered.
focals = [500.0, 850.0, 1400.0]

fig = plt.figure(figsize=(15.5, 4.9), layout="constrained")
gs = fig.add_gridspec(1, 4, width_ratios=[1.5, 1.0, 1.0, 1.0])

ax = fig.add_subplot(gs[0], projection="3d")
for q in Xc:
    ax.plot(*zip(np.zeros(3), q), lw=0.6, color=GREY, alpha=.8)
wireframe_3d(ax, Xc, dots=False)
draw_eye(ax, np.zeros(3))
scale = Xc[:, 2].min() / max(focals)
for fx in focals:
    zp = fx * scale
    hitp = zp * Xc[:, :2] / Xc[:, 2, None]
    h = 1.35 * np.abs(hitp).max()
    gx, gy = np.meshgrid([-h, h], [-h, h])
    ax.plot_surface(gx, gy, np.full_like(gx, zp), alpha=.13, color=ACCENT)
    for a, b in EDGES:
        q = hitp[[index[a], index[b]]]
        ax.plot(q[:, 0], q[:, 1], np.full(2, zp), lw=1.2, color=ACCENT)
ax.set_title("one pyramid, three planes", fontsize=10)
ax.view_init(elev=9, azim=-70); set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.65); ax.set_axis_off()

for j, fx in enumerate(focals):
    ax = fig.add_subplot(gs[j + 1])
    K_f = K_matrix(fx=fx, cx=W_PIX/2, cy=H_PIX/2)
    pic, _ = project_points(X, K_f, R, t)
    wireframe_2d(ax, pic, rf"$f_x=f_y={fx:.0f}$ px", W_PIX, H_PIX, dots=False)
    image_frame(ax, W_PIX, H_PIX); ax.axis("off")
plt.show()

In [ ]:
#| echo: false
#| column: page
#| label: fig-principal-point
#| fig-cap: >-
#|   The principal point, with the focal length fixed. The orange cross marks
#|   $(c_x,c_y)$, where the optical axis meets the sensor. Changing it slides the
#|   picture inside the image rectangle without altering its shape: the principal
#|   point is a property of how the sensor is placed and cropped, not of the
#|   pyramid. This is why a cropped photograph has a principal point away from
#|   the centre of its raster.
centres = [(W_PIX/2, H_PIX/2), (W_PIX/2 - 190, H_PIX/2 - 90),
           (W_PIX/2 + 150, H_PIX/2 + 110)]

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.4), layout="constrained")
for ax, (cx, cy) in zip(axes, centres):
    K_c = K_matrix(fx=850, cx=cx, cy=cy)
    pic, _ = project_points(X, K_c, R, t)
    wireframe_2d(ax, pic, rf"$(c_x,c_y)=({cx:.0f},{cy:.0f})$", W_PIX, H_PIX,
                 dots=False)
    ax.scatter([cx], [cy], marker="+", s=140, lw=2.0, color=ACCENT, zorder=6)
    image_frame(ax, W_PIX, H_PIX); ax.axis("off")
plt.show()

## The camera matrix

Combining the normalised projection

$$
\lambda\mathbf x_n
=
\begin{bmatrix} \mathsf I & \mathbf 0 \end{bmatrix}\mathbf{X}_c
$$

with the calibration transformation gives
$\lambda\mathbf x=\mathsf K[\mathsf I\mid\mathbf 0]\mathbf{X}_c$, and if the
scene is expressed in world coordinates,

$$
\lambda\mathbf x
=
\mathsf K[\mathsf R\mid\mathbf t]\,\mathbf X_w ,
\qquad
\mathsf P=\mathsf K[\mathsf R\mid\mathbf t] .
$$

The two factors have different roles: $\mathsf K$ describes the **internal
geometry of the camera**, while $\mathsf R$ and $\mathbf t$ describe its **pose
in the world**. Counting parameters, five for $\mathsf K$ and six for the pose
makes eleven, which is exactly the number of degrees of freedom of a $3\times4$
matrix defined up to scale.

In [ ]:
#| echo: false
#| column: page
#| label: fig-three-maps
#| fig-cap: >-
#|   The same house at the four stages of the chain. **Top left:** the world
#|   frame, with the camera drawn where it sits and three rays showing what it is
#|   looking at. **Top right:** after the rigid motion, the identical object in
#|   the camera's own frame — nothing projected yet, only relabelled. **Bottom
#|   left:** after the interception, in normalised coordinates, whose units are
#|   dimensionless ratios. **Bottom right:** after $\mathsf{K}$, in pixels, which
#|   is what a sensor reports.
p_norm = Xc[:, :2] / Xc[:, 2, None]

fig = plt.figure(figsize=(13, 9.6), layout="constrained")

ax = fig.add_subplot(221, projection="3d")
wireframe_3d(ax, X, dots=False)
frame_3d(ax, C, R.T, [r"$x_c$", r"$y_c$", r"$z_c$"], color=ACCENT, length=3.0)
frame_3d(ax, np.zeros(3), np.eye(3), [r"$x_w$", r"$y_w$", r"$z_w$"],
         color=DARK, length=3.0)
draw_eye(ax, C)
for k in [IDS[0], IDS[-1], IDS[len(IDS)//2]]:
    ax.plot(*zip(C, X[index[k]]), lw=0.9, color=GREY)
ax.set_title("the world frame", fontsize=10)
ax.view_init(elev=22, azim=-55); set_axes_equal(ax); ax.set_axis_off()

ax = fig.add_subplot(222, projection="3d")
wireframe_3d(ax, Xc, dots=False)
frame_3d(ax, np.zeros(3), np.eye(3), [r"$x_c$", r"$y_c$", r"$z_c$"],
         color=ACCENT, length=4.0)
draw_eye(ax, np.zeros(3))
ax.set_title(r"after $[\mathsf{R}\mid\mathbf{t}]$: the camera frame", fontsize=10)
ax.view_init(elev=18, azim=-60); set_axes_equal(ax); ax.set_axis_off()

ax = fig.add_subplot(223)
wireframe_2d(ax, p_norm, r"after $[\mathsf{I}\mid\mathbf{0}]$: normalised",
             dots=False)
ax.set_xlabel("$X_c/Z_c$"); ax.set_ylabel("$Y_c/Z_c$")

ax = fig.add_subplot(224)
wireframe_2d(ax, xy, r"after $\mathsf{K}$: pixels", W_PIX, H_PIX, dots=False)
image_frame(ax, W_PIX, H_PIX)
ax.set_xlabel("$u$ [px]"); ax.set_ylabel("$v$ [px]")
plt.show()

## Back-projecting a pixel

The model can also be read backwards, and the reading is short.

Applying $\mathsf K^{-1}$ to a pixel $\mathbf x=(u,v,1)^\top$ strips off the
image coordinate system and returns the direction of the corresponding visual
ray in the camera frame,

$$
\mathbf d_c \propto \mathsf K^{-1}\mathbf x
=
\bigl((u-c_x)/f_x,\;(v-c_y)/f_y,\;1\bigr)^\top
\quad\text{when } s=0 .
$$

Rotating that direction back into the world and starting from the camera centre
gives the whole ray:

$$
\mathbf X_w(\mu)=\mathbf C+\mu\,\mathsf R^\top\mathsf K^{-1}\mathbf x ,
\qquad \mu>0 .
$$

So the extrinsics contribute two different things: $\mathbf C$ says where the ray
starts, $\mathsf R$ says which way it points. And calibration turns a pixel into
a **direction** — not into a point. Where the scene lies along the ray is exactly
the information the projection destroyed, and no amount of calibration will bring
it back.

A useful consequence is that a calibrated camera measures angles. The angle
between the rays of two pixels is

$$
\cos\theta=
\frac{\mathbf d_1^\top\mathbf d_2}{\lVert\mathbf d_1\rVert\,\lVert\mathbf d_2\rVert},
\qquad
\mathbf d_i=\mathsf K^{-1}\mathbf x_i ,
$$

and the pose never enters, because a rotation does not change angles.

In [ ]:
def backproject(pixels, K, R, t):
    """World-frame origin and unit directions of the rays of a set of pixels."""
    x_h = np.c_[np.asarray(pixels, float), np.ones(len(pixels))]
    d_c = (np.linalg.inv(K) @ x_h.T).T
    d_w = (R.T @ d_c.T).T
    return -R.T @ t, d_w / np.linalg.norm(d_w, axis=1, keepdims=True)


picked = [IDS[0], IDS[len(IDS)//2], IDS[-1]]
pix = xy[[index[k] for k in picked]]
origin, dirs = backproject(pix, K, R, t)

print("each ray passes through the vertex that produced it:")
for k, d in zip(picked, dirs):
    v = X[index[k]] - origin
    gap = np.linalg.norm(v - (v @ d) * d)        # distance from vertex to ray
    print(f"  {k}:  {gap:.2e} cm off the ray")

d1 = np.linalg.solve(K, np.append(pix[0], 1.0))
d2 = np.linalg.solve(K, np.append(pix[-1], 1.0))
cos = d1 @ d2 / (np.linalg.norm(d1) * np.linalg.norm(d2))
print(f"\nangle between the first and last ray, from the pixels alone: "
      f"{np.degrees(np.arccos(np.clip(cos, -1, 1))):.4f} deg")

In [ ]:
#| echo: false
#| column: page
#| label: fig-backprojection
#| fig-cap: >-
#|   The model running backwards. **Left:** three chosen pixels in the digital
#|   image. **Right:** the rays they define in the world, drawn from the camera
#|   centre and continued a little past the house, following
#|   $\mathbf{X}_w(\mu)=\mathbf{C}+\mu\,\mathsf{R}^\top\mathsf{K}^{-1}\mathbf{x}$.
#|   Each ray meets the vertex that produced the pixel, but calibration alone does
#|   not say where along the ray that vertex lies.
cols = [BLUE, RED, "#7B3294"]

fig = plt.figure(figsize=(14.0, 5.2), layout="constrained")
gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.15])

ax = fig.add_subplot(gs[0])
wireframe_2d(ax, xy, "three pixels in the image", W_PIX, H_PIX, color=GREY,
             lw=1.4,
             dots=False)
image_frame(ax, W_PIX, H_PIX)
for (u, v), col, k in zip(pix, cols, picked):
    ax.scatter([u], [v], s=90, color=col, zorder=6)
    ax.text(u + 18, v - 16, k, fontsize=10, color=col)
ax.axis("off")

ax = fig.add_subplot(gs[1], projection="3d")
wireframe_3d(ax, X, dots=False)
# Each ray is drawn a little past the vertex that produced it, so that the
# figure stays framed on the house instead of on the rays.
for d, col, k in zip(dirs, cols, picked):
    reach = 1.18 * np.linalg.norm(X[index[k]] - origin)
    seg = np.vstack([origin, origin + reach*d])
    ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], lw=1.6, color=col)
    ax.scatter(*X[index[k]], s=60, color=col, zorder=7)
draw_eye(ax, origin, r"$\mathbf{C}$", offset=[0, 0, -2.0])
ax.set_title("their rays in the world", fontsize=10)
ax.view_init(elev=18, azim=-52); set_axes_equal(ax)
ax.set_box_aspect((1, 1, 1), zoom=1.65); ax.set_axis_off()
plt.show()

If only the matrix $\mathsf P$ is known, the ray can still be recovered, but it
comes out as a projective line rather than as a Euclidean ray with an origin and
a direction.

The centre is the right null space, $\mathsf P\mathbf C=\mathbf 0$. The
pseudoinverse supplies one preimage of the pixel, $\mathbf X_0=\mathsf
P^{+}\mathbf x$, and since adding any multiple of the centre leaves the image
unchanged, the full back-projection is the line

$$
\mathbf X\sim\alpha\,\mathsf P^{+}\mathbf x+\beta\,\mathbf C .
$$

The pseudoinverse is not an inverse camera: it picks one point on the line, and
the null space supplies the rest.

## Reading the camera back out of $\mathsf P$

Sometimes we are given only the $3\times4$ matrix. Its geometry can be read back
out. The camera centre is the right null vector, because the optical centre has
no image. If the world frame is Euclidean and the matrix really represents a
finite pinhole camera, the left $3\times3$ block factorises by an **RQ
decomposition**,

$$
\mathsf M=\mathsf K\mathsf R ,
$$

recovering intrinsics and rotation up to the usual sign conventions; then
$\mathbf t$ follows from the fourth column. With $K_{33}=1$, $K_{11},K_{22}>0$
and $\det\mathsf R=1$ the factorisation is unique.

RQ is not a calibration algorithm: it only decomposes the matrix it is given. If $\mathsf P$ has been estimated
poorly, its factors will be poor as well. Calibration procedures estimate the
camera while enforcing its structure during the estimation, and using many
measurements.

In [ ]:
def rq3(A):
    """RQ decomposition A = R_upper @ Q_orthogonal, built from NumPy's QR."""
    Q, R_up = np.linalg.qr(np.flipud(A).T)
    return np.fliplr(np.flipud(R_up.T)), np.flipud(Q.T)


def decompose_projection_matrix(Pm):
    """Recover K, R, t from a finite Euclidean camera matrix."""
    Pw = np.asarray(Pm, float).copy()
    if np.linalg.det(Pw[:, :3]) < 0:
        Pw = -Pw
    K_hat, R_hat = rq3(Pw[:, :3])

    signs = np.sign(np.diag(K_hat)); signs[signs == 0] = 1.0
    D = np.diag(signs)
    K_hat, R_hat = K_hat @ D, D @ R_hat

    K_hat, Pw = K_hat / K_hat[2, 2], Pw / K_hat[2, 2]
    return K_hat, R_hat, np.linalg.solve(K_hat, Pw[:, 3])


# The centre, from P alone: no Euclidean assumption is used here.
M, p4 = P[:, :3], P[:, 3]
C_from_P = -np.linalg.solve(M, p4)
print(f"centre from P, error against the true one: "
      f"{np.max(np.abs(C_from_P - C)):.2e} cm")

K_hat, R_hat, t_hat = decompose_projection_matrix(P)
print(f"\nRQ on an exact P:  |K-K_hat| = {np.max(np.abs(K_hat - K)):.2e}   "
      f"|R-R_hat| = {np.max(np.abs(R_hat - R)):.2e}   "
      f"|t-t_hat| = {np.max(np.abs(t_hat - t)):.2e}")

## Further reading

- Hartley, R. and Zisserman, A. *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press, 2004. Chapter 6 for the projective camera and its decomposition.
- Fusiello, A. *Visione Computazionale: tecniche di ricostruzione tridimensionale*, Franco Angeli, 2018, for a geometric treatment of perspective cameras and projective coordinates.
- Alberti, L. B. *De pictura* / *Della pittura*, 1435–36, for the visual pyramid and the *velo*. Cecil Grayson's is the standard critical edition.
- Andersen, K. *The Geometry of an Art*, Springer, 2007, for the historical development of mathematical perspective.

---

**Luca Magri** — Computer Vision Dojo  
Code MIT · text and figures CC BY-NC-ND 4.0  
<https://magrilu.github.io/cv-dojo/>